# Financial Data Quality Audit

This notebook validates the datasets declared in `reports/data_inventory.yml` and checks if files are present, readable, and structurally consistent with metadata.

## Checks included

1. Inventory schema sanity checks
2. Structural QA (file, sheet, metadata, header/date integrity)
3. ETL smoke test (extract tabular data from each workbook)
4. EDA sanity metrics (rows, date range, duplicates, sorting, missing ratio)
5. Consolidated critical issues vs expected warnings
6. Export of detailed QA tables

In [41]:
from pathlib import Path
import warnings

import pandas as pd
import yaml
from openpyxl import load_workbook

warnings.filterwarnings('ignore', category=UserWarning)

In [42]:
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

INV_PATH = ROOT / 'reports' / 'data_inventory.yml'
assert INV_PATH.exists(), f'Inventory not found: {INV_PATH}'

with INV_PATH.open('r', encoding='utf-8') as f:
    inv = yaml.safe_load(f)

datasets = inv.get('datasets', [])
df = pd.DataFrame(datasets)
print(f'inventory_file={INV_PATH}')
print(f'datasets={len(df)}')
df.head(3)

inventory_file=C:\Users\Jorge\OneDrive - UFV\BAINF\PAPER\reports\data_inventory.yml
datasets=46


,id,path,category,subcategory,ticker,sheet,header_row,data_start_row,freq,variables,unit,status,notes
0,investable_assets_cash_euribor_3m,data/investable_assets/cash/EURIBOR_3M.xlsx,investable_assets,cash,EURIBOR_3M,Sheet 1,10.0,11.0,daily,"[Exchange Date, Fixing Value]",rate_level,OK,
1,investable_assets_commodities_bloomberg_commodity,data/investable_assets/commodities/Bloomberg_C...,investable_assets,commodities,Bloomberg_Commodity,Sheet 1,31.0,32.0,daily,"[Exchange Date, Close, Net, %Chg, Open, Low, H...",price_index_level,OK,
2,investable_assets_commodities_brent,data/investable_assets/commodities/Brent.xlsx,investable_assets,commodities,Brent,Sheet 1,34.0,35.0,daily,"[Exchange Date, Close, Net, %Chg, Open, Low, H...",price_index_level,OK,


In [43]:
required_cols = [
    'id', 'path', 'category', 'subcategory', 'ticker', 'sheet',
    'header_row', 'data_start_row', 'freq', 'variables', 'unit', 'status'
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns in inventory: {missing_cols}')

df['path_obj'] = df['path'].map(lambda p: ROOT / p if isinstance(p, str) else None)
df['file_exists'] = df['path_obj'].map(lambda p: p.exists() if p is not None else False)
df['is_xlsx'] = df['path'].map(lambda p: isinstance(p, str) and p.lower().endswith('.xlsx'))

print('schema_ok=True')
print(df[['status', 'file_exists']].value_counts(dropna=False).rename('count'))

schema_ok=True
status               file_exists
OK                   True           41
EMPTY_METADATA_ONLY  True            2
NEEDS_REFRESH        True            2
WRONG_ENTITY         True            1
Name: count, dtype: int64


In [ ]:
def safe_cell_value(ws, row, col=1):
    if row is None or row < 1:
        return None
    return ws.cell(row=row, column=col).value

def normalize_row_index(x):
    if pd.isna(x):
        return None
    try:
        x_num = float(x)
        if x_num.is_integer() and x_num > 0:
            return int(x_num)
    except Exception:
        return None
    return None

def parse_date_safe(x):
    if x is None or x == '':
        return pd.NaT
    if isinstance(x, (int, float)) and not isinstance(x, bool):
        if x > 59:
            dt = pd.to_datetime(x, errors='coerce', unit='D', origin='1899-12-30')
            if pd.notna(dt):
                return dt
    return pd.to_datetime(x, errors='coerce')

def first_numeric_after_date(row_values):
    for v in row_values[1:]:
        if isinstance(v, (int, float)) and not isinstance(v, bool):
            return float(v)
    return None

def has_live_query_formula(ws, max_rows=40, max_cols=12):
    for r in range(1, min(ws.max_row, max_rows) + 1):
        for c in range(1, max_cols + 1):
            v = ws.cell(r, c).value
            if isinstance(v, str) and v.startswith('=') and ('RDP.' in v or 'TR.' in v or 'HistoricalPricing' in v):
                return True
    return False

def infer_data_start_row(ws, preferred_hint=None, scan_limit=250):
    candidates = []
    if preferred_hint is not None:
        candidates.append(preferred_hint)
    candidates.extend(range(1, min(scan_limit, ws.max_row) + 1))

    seen = set()
    ordered = []
    for c in candidates:
        if c not in seen:
            ordered.append(c)
            seen.add(c)

    for r in ordered:
        raw = safe_cell_value(ws, r, 1)
        dt = parse_date_safe(raw)
        if pd.notna(dt):
            return r
    return None

def extract_timeseries(ws, data_start_hint=None, row_cap=8000):
    dr = infer_data_start_row(ws, preferred_hint=data_start_hint)
    if dr is None:
        return dr, [], []

    dates = []
    values = []
    blank_streak = 0

    for row in ws.iter_rows(min_row=dr, values_only=True):
        raw_date = row[0] if len(row) > 0 else None
        dt = parse_date_safe(raw_date)

        if pd.isna(dt):
            blank_streak += 1
            if blank_streak >= 10 and len(dates) > 0:
                break
            continue

        blank_streak = 0
        dates.append(dt)
        values.append(first_numeric_after_date(row))

        if len(dates) >= row_cap:
            break

    return dr, dates, values

def evaluate_sheet(ws, data_start_hint=None):
    dr_used, dates, values = extract_timeseries(ws, data_start_hint=data_start_hint)
    return {
        'dr_used': dr_used,
        'dates': dates,
        'values': values,
        'n_rows': len(dates),
    }

skip_ids = {'regime_variables_monetary_ecb_balance_sheet'}
skip_statuses = {'WRONG_ENTITY'}
expected_incomplete_statuses = {'EMPTY_METADATA_ONLY', 'NEEDS_REFRESH'}

rows = []
for rec in df.to_dict(orient='records'):
    status = rec.get('status', 'OK')
    if rec.get('id') in skip_ids or status in skip_statuses:
        continue

    hr = normalize_row_index(rec.get('header_row'))
    dr = normalize_row_index(rec.get('data_start_row'))

    out = {
        'id': rec['id'],
        'path': rec['path'],
        'status': status,
        'sheet_expected': rec.get('sheet'),
        'sheet_used': rec.get('sheet'),
        'header_row_expected': hr,
        'data_start_row_expected': dr,
        'data_start_row_used': None,
        'n_variables_expected': len(rec.get('variables') or []),
        'file_exists': False,
        'workbook_ok': False,
        'sheet_ok': False,
        'header_row_has_values': False,
        'n_header_cells_detected': 0,
        'first_date_parse_ok': None,
        'first_date_value': None,
        'n_rows_extracted': 0,
        'n_valid_dates': 0,
        'date_min': None,
        'date_max': None,
        'date_is_monotonic': None,
        'n_duplicate_dates': None,
        'value_missing_ratio': None,
        'issues': [],
        'warnings': []
    }

    p = ROOT / rec['path']
    if not p.exists():
        out['issues'].append('missing_file')
        rows.append(out)
        continue

    out['file_exists'] = True

    try:
        wb = load_workbook(filename=p, data_only=True, read_only=True)
        out['workbook_ok'] = True
    except Exception:
        out['issues'].append('cannot_open_workbook')
        rows.append(out)
        continue

    expected_sheet = rec.get('sheet')
    if expected_sheet not in wb.sheetnames:
        out['issues'].append('missing_sheet')
        rows.append(out)
        continue

    ws_expected = wb[expected_sheet]
    out['sheet_ok'] = True

    if hr is None:
        if status in expected_incomplete_statuses:
            out['warnings'].append('expected_missing_header_row_metadata')
        else:
            out['issues'].append('missing_header_row_metadata')

    if dr is None:
        if status in expected_incomplete_statuses:
            out['warnings'].append('expected_missing_data_start_row_metadata')
        else:
            out['issues'].append('missing_data_start_row_metadata')

    if hr is not None:
        try:
            header_values = list(next(ws_expected.iter_rows(min_row=hr, max_row=hr, values_only=True)))
            non_empty_header = [v for v in header_values if v not in (None, '')]
            out['n_header_cells_detected'] = len(non_empty_header)
            out['header_row_has_values'] = len(non_empty_header) > 0
            if not out['header_row_has_values']:
                if status in expected_incomplete_statuses:
                    out['warnings'].append('expected_empty_header_row')
                else:
                    out['issues'].append('empty_header_row')

            n_expected = out['n_variables_expected']
            if n_expected > 0 and len(non_empty_header) < n_expected:
                out['warnings'].append('header_shorter_than_expected_variables')
        except Exception:
            out['issues'].append('header_read_error')

    # ETL/EDA extraction with sheet fallback when expected sheet has no cached rows.
    extraction = evaluate_sheet(ws_expected, data_start_hint=dr)
    best_sheet_name = expected_sheet
    best_extraction = extraction

    if extraction['n_rows'] == 0:
        fallback_priority = ['First Release Data', 'Table Data', 'VIX_History']
        candidate_sheets = [s for s in fallback_priority if s in wb.sheetnames and s != expected_sheet]
        candidate_sheets.extend([s for s in wb.sheetnames if s not in candidate_sheets and s != expected_sheet])

        for s in candidate_sheets:
            ws_alt = wb[s]
            alt = evaluate_sheet(ws_alt, data_start_hint=None)
            if alt['n_rows'] > best_extraction['n_rows']:
                best_extraction = alt
                best_sheet_name = s

    out['sheet_used'] = best_sheet_name
    out['data_start_row_used'] = best_extraction['dr_used']
    out['n_rows_extracted'] = best_extraction['n_rows']
    out['n_valid_dates'] = best_extraction['n_rows']

    if best_extraction['n_rows'] == 0:
        ws_for_formula = wb[expected_sheet]
        live_formula = has_live_query_formula(ws_for_formula)

        if live_formula:
            out['warnings'].append('live_query_formula_without_cached_history')
        elif status in expected_incomplete_statuses:
            out['warnings'].append('expected_no_cached_data_rows')
        else:
            out['issues'].append('no_data_rows_extracted')
    else:
        ds = pd.Series(best_extraction['dates'], dtype='datetime64[ns]')
        out['date_min'] = ds.min()
        out['date_max'] = ds.max()
        monotonic_inc = bool(ds.is_monotonic_increasing)
        monotonic_dec = bool(ds.is_monotonic_decreasing)
        out['date_is_monotonic'] = monotonic_inc or monotonic_dec
        out['n_duplicate_dates'] = int(ds.duplicated().sum())

        val_series = pd.Series(best_extraction['values'], dtype='float64')
        out['value_missing_ratio'] = float(val_series.isna().mean())

        out['first_date_value'] = str(best_extraction['dates'][0])
        out['first_date_parse_ok'] = True

        if out['n_duplicate_dates'] > 0:
            out['warnings'].append('duplicate_dates')
        if not out['date_is_monotonic']:
            out['warnings'].append('date_not_sorted')
        if out['value_missing_ratio'] is not None and out['value_missing_ratio'] > 0.25:
            out['warnings'].append('high_missing_ratio_value_series')
        if best_sheet_name != expected_sheet:
            out['warnings'].append('used_fallback_sheet_for_extraction')

    out['issue_count'] = len(out['issues'])
    out['warning_count'] = len(out['warnings'])
    out['quality_ok'] = out['issue_count'] == 0
    rows.append(out)

qa = pd.DataFrame(rows)
qa = qa.sort_values(['issue_count', 'warning_count', 'id'], ascending=[False, False, True]).reset_index(drop=True)
print(f'skipped_wrong_entity={len(skip_ids)}')
qa.head(8)

In [38]:
total = len(qa)
ok = int(qa['quality_ok'].sum())
fail = total - ok
score = round(100 * ok / total, 2) if total else 0.0

print(f'total_datasets={total}')
print(f'critical_ok={ok}')
print(f'critical_fail={fail}')
print(f'critical_score_pct={score}')
print(f'total_warnings={int(qa["warning_count"].sum())}')

issue_table = (
    qa.explode('issues')
      .dropna(subset=['issues'])
      .groupby('issues', as_index=False)
      .size()
      .sort_values('size', ascending=False)
)

warning_table = (
    qa.explode('warnings')
      .dropna(subset=['warnings'])
      .groupby('warnings', as_index=False)
      .size()
      .sort_values('size', ascending=False)
)

print('\nCritical issues:')
display(issue_table if len(issue_table) else pd.DataFrame({'issues': [], 'size': []}))
print('\nWarnings:')
display(warning_table if len(warning_table) else pd.DataFrame({'warnings': [], 'size': []}))

total_datasets=45
critical_ok=45
critical_fail=0
critical_score_pct=100.0
total_warnings=6

Critical issues:


,issues,size



Warnings:


,warnings,size
0,expected_missing_data_start_row_metadata,2
1,expected_missing_header_row_metadata,2
2,expected_no_cached_data_rows,2


In [39]:
problematic = qa.loc[(qa['issue_count'] > 0) | (qa['warning_count'] > 0), [
    'id', 'path', 'status', 'sheet_expected', 'header_row_expected',
    'data_start_row_expected', 'n_rows_extracted', 'date_min', 'date_max',
    'n_duplicate_dates', 'value_missing_ratio', 'issue_count', 'warning_count', 'issues', 'warnings'
]]
problematic = problematic.sort_values(['issue_count', 'warning_count', 'status', 'id'], ascending=[False, False, True, True])
problematic

,id,path,status,sheet_expected,header_row_expected,data_start_row_expected,n_rows_extracted,date_min,date_max,n_duplicate_dates,value_missing_ratio,issue_count,warning_count,issues,warnings
0,regime_variables_inflation_eurozone_ppi,data/regime_variables/inflation/Eurozone_PPI.xlsx,EMPTY_METADATA_ONLY,Historical Values,NaN,NaN,0,NaT,NaT,NaN,NaN,0,2,[],"[expected_missing_header_row_metadata, expecte..."
1,regime_variables_monetary_germany_2y_yield,data/regime_variables/monetary/Germany_2Y_Yiel...,EMPTY_METADATA_ONLY,Historical Values,NaN,NaN,0,NaT,NaT,NaN,NaN,0,2,[],"[expected_missing_header_row_metadata, expecte..."
2,regime_variables_inflation_germany_hicp,data/regime_variables/inflation/Germany_HICP.xlsx,NEEDS_REFRESH,Historical Values,20.0,21.0,0,NaT,NaT,NaN,NaN,0,1,[],[expected_no_cached_data_rows]
3,regime_variables_sentiment_zew_germany,data/regime_variables/sentiment/ZEW_Germany.xlsx,NEEDS_REFRESH,Historical Values,19.0,20.0,0,NaT,NaT,NaN,NaN,0,1,[],[expected_no_cached_data_rows]


In [40]:
OUT_DIR = ROOT / 'reports' / 'quality'
OUT_DIR.mkdir(parents=True, exist_ok=True)

detailed_path = OUT_DIR / 'data_quality_detailed.csv'
issues_path = OUT_DIR / 'data_quality_issues.csv'
summary_path = OUT_DIR / 'data_quality_issue_summary.csv'
warning_summary_path = OUT_DIR / 'data_quality_warning_summary.csv'

qa.to_csv(detailed_path, index=False)
problematic.to_csv(issues_path, index=False)
issue_table.to_csv(summary_path, index=False)
warning_table.to_csv(warning_summary_path, index=False)

print(f'written: {detailed_path}')
print(f'written: {issues_path}')
print(f'written: {summary_path}')
print(f'written: {warning_summary_path}')

written: C:\Users\Jorge\OneDrive - UFV\BAINF\PAPER\reports\quality\data_quality_detailed.csv
written: C:\Users\Jorge\OneDrive - UFV\BAINF\PAPER\reports\quality\data_quality_issues.csv
written: C:\Users\Jorge\OneDrive - UFV\BAINF\PAPER\reports\quality\data_quality_issue_summary.csv
written: C:\Users\Jorge\OneDrive - UFV\BAINF\PAPER\reports\quality\data_quality_warning_summary.csv


## Interpretation guide

- `issues`: critical blockers for production modeling (must fix).
- `warnings`: expected gaps or soft-quality signals (review, but not always blocking).
- `expected_missing_*`: typical for `NEEDS_REFRESH` / `EMPTY_METADATA_ONLY` datasets.
- `no_data_rows_extracted`: ETL could not pull usable timeseries rows.
- `duplicate_dates` / `date_not_sorted`: index quality problems for time-series pipelines.
- `high_missing_ratio_value_series`: too many null numeric observations in extracted series.

Use this notebook as ETL + EDA pre-flight validation before running empirical models.